# NB16 — Feature Engineering + Stacking (Panel-Bazli)

**TEKNOFEST Saglikta Yapay Zeka | Genetik Varyant Patojenite Tahmini**

NB15'in panel-bazli kurgusunu (S1 = panel %50 + dengelenmis MASTER cekirdek) temel alir,
uzerine **iki yeni katman** ekler:

## 1. Literatur-destekli Feature Engineering (az ama anlamli)

Literatur taramasi (REVEL, Grantham/BLOSUM62 calismalari) isiginda, anonim veride
**guvenli ve biyolojik olarak gerekceli** az sayida ozellik:

| Ozellik | Gerekce (literatur) |
|---|---|
| `aa_stopgain` (`AA_2=='*'`) | Stop-gain/nonsense mutasyonu — guclu patojenite sinyali |
| `aa_nonstandard` | AA degeri standart 20 disi (indel/frameshift kalintisi) |
| `grantham_distance` | Amino asit yan zincir farki; patojeniklerde ortalama yuksek |
| `blosum62_score` | Evrimsel substitusyon toleransi (negatif = nadir/zararli) |
| `al_freq_log` | Genis-aralikli AL frekans sutunlarina log1p (orijinal korunur) |

FE **dikkatli tutuldu**: ayni anda cok sey degistirmemek icin yalnizca standart,
deterministik turetmeler. K-mer / agresif etkilesim YOK.

## 2. OOF Stacking (literatur protokolu)

Out-of-fold (OOF) stacking, meta-learner'i base modellerin **leakage'siz** tahminleriyle
egitir. NB15'teki dort model tipi base olur:

- Base: **LightGBM, CatBoost, NN (SmallMLP), DNN** — her biri 5-fold OOF olasilik uretir.
- Meta-feature: 4 OOF olasilik + cesitlilik (mean/std).
- Meta-learner: **Logistic Regression** (L2, class_weight=balanced) — literaturde en
  yaygin/kararli meta secimi.

## Degerlendirme (NB15 v2 ile ayni)

- Threshold UC modda: `f1_raw` / `f1_8020` / `mcc_8020` (final %80/20 benign-aware).
- Test: %50/50 + **bootstrap %80/20** (final dagilim, %95 CI). Birincil = %80/20 F1.
- Test'te MASTER YOK. Imputer/encoder/scaler yalniz train'de fit (leakage-free).


## v2 GUNCELLEMESI (FE ablasyonu + 6 base + 2 meta + train sonuclari)

NB15 bulgularina gore genisletildi:
- **FE ablasyonu**: her sey iki kez kosar — **FE-yok** (sadece M3, NB15 ile ayni) ve
  **FE-var** (M3 + Grantham/BLOSUM62/stopgain). Boylece FE'nin %80/20 F1'e net katkisi
  izole olcular ("ayni anda cok sey degistirme" prensibi).
- **6 base model**: scratch (lightgbm, catboost, nn, dnn) + **finetune (nn_ft, dnn_ft)**
  — NB15'te KANSER'i finetune-DNN kazanmisti; stacking en iyi modelleri icermeli.
- **2 meta-learner yan yana**: Logistic Regression + LightGBM (ikisi de class_weight=balanced).
- **Train sonuclari** raporda (overfit kontrolu; NB15 v1'de eksikti).


In [1]:
# Cell 1: Imports & Config
import os, sys, warnings, json
from copy import deepcopy
from datetime import datetime
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (f1_score, precision_score, recall_score,
                             confusion_matrix, matthews_corrcoef)
warnings.filterwarnings("ignore")

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if not os.path.exists(os.path.join(PROJECT_ROOT, "config.py")):
    PROJECT_ROOT = os.path.abspath(os.getcwd())
sys.path.insert(0, PROJECT_ROOT)

from config import SEED, REPORTS_DIR
from src import columns_real as CR
from src.models import (MLP3Layer, DeepMLP, LGBM_GRID, LGBM_FIXED, CB_GRID, CB_FIXED)
from src.metrics import optimize_threshold
from src.focal_loss import FocalLoss
import lightgbm as lgb
from catboost import CatBoostClassifier

np.random.seed(SEED); torch.manual_seed(SEED)
try:
    import torch_directml; DEVICE = torch_directml.device()
except Exception:
    DEVICE = torch.device("cpu")

PANELS = ["CFTR", "KANSER", "PAH"]
MASTER_CORE_POS = MASTER_CORE_NEG = 625
HIGH_MISSING_THRESHOLD = 0.50
PANEL_SPLIT_FRAC = 0.50
N_OOF = 5                    # OOF fold sayisi
FINAL_BENIGN_FRAC = 0.80
N_BOOT = 50
BOOT_SEED = SEED

DATA_DIR = os.path.join(PROJECT_ROOT, "data", "real_data")
RESULTS_DIR = os.path.join(PROJECT_ROOT, "results", "v6_fe_stacking")
os.makedirs(RESULTS_DIR, exist_ok=True); os.makedirs(REPORTS_DIR, exist_ok=True)
print(f"PROJECT_ROOT: {PROJECT_ROOT}  DEVICE: {DEVICE}  SEED: {SEED}")
print(f"OOF fold: {N_OOF}  final benign: {int(FINAL_BENIGN_FRAC*100)}%  bootstrap N: {N_BOOT}")


PROJECT_ROOT: c:\Users\ahmet.ceyhan23\Desktop\teknofest_model  DEVICE: privateuseone:0  SEED: 42
OOF fold: 5  final benign: 80%  bootstrap N: 50


In [2]:
# Cell 2: Veri Yukleme + cross-panel birebir-ayni satir drop (NB15 ile ayni)
ID_COL, TARGET = CR.ID_COL, CR.TARGET_COL
def load_panel(name): return pd.read_csv(os.path.join(DATA_DIR, CR.PANEL_INFO[name]["file"]))
master = load_panel("MASTER")
panels_raw = {p: load_panel(p) for p in PANELS}
feature_cols_all = [c for c in master.columns if c not in (ID_COL, TARGET)]

master_index = {}
for _, row in master.iterrows():
    key = (row[ID_COL],) + tuple((np.nan if pd.isna(v) else v) for v in row[feature_cols_all].values)
    master_index[key] = row[TARGET]
panels, dup_report = {}, {}
for p, df in panels_raw.items():
    drop_idx = [idx for idx, row in df.iterrows()
                if (((row[ID_COL],) + tuple((np.nan if pd.isna(v) else v)
                     for v in row[feature_cols_all].values)) in master_index
                    and master_index[(row[ID_COL],) + tuple((np.nan if pd.isna(v) else v)
                         for v in row[feature_cols_all].values)] == row[TARGET])]
    panels[p] = df.drop(index=drop_idx).reset_index(drop=True)
    dup_report[p] = len(drop_idx)
print("MASTER:", master.shape, master[TARGET].value_counts().to_dict())
for p, df in panels.items():
    print(f"{p}: {df.shape} {df[TARGET].value_counts().to_dict()} (drop={dup_report[p]})")


MASTER: (2931, 353) {1: 2149, 0: 782}
CFTR: (111, 353) {1: 90, 0: 21} (drop=0)
KANSER: (385, 353) {1: 265, 0: 120} (drop=3)
PAH: (369, 353) {1: 307, 0: 62} (drop=3)


In [3]:
# Cell 3: Sutun temizligi (sabit + ozdes) — NB15 ile ayni global feature listesi
constant_cols = CR.get_constant_cols(master[feature_cols_all])
dup_pairs = CR.get_duplicate_col_pairs(master[feature_cols_all])
dup_drop = sorted({b for (a, b) in dup_pairs})
drop_cols = sorted(set(constant_cols) | set(dup_drop))
base_feature_cols = [c for c in feature_cols_all if c not in drop_cols]
CAT_LIKE = [c for c in (CR.CAT_COLS + CR.AA_COLS) if c in base_feature_cols]
NUM_COLS_BASE = [c for c in base_feature_cols if c not in CAT_LIKE]
print(f"drop {len(drop_cols)} sutun -> {len(base_feature_cols)} ham feature "
      f"({len(NUM_COLS_BASE)} sayisal + {len(CAT_LIKE)} kategorik)")


drop 63 sutun -> 288 ham feature (281 sayisal + 7 kategorik)


In [4]:
# Cell 4: Feature Engineering (literatur-destekli, deterministik)
# Grantham mesafesi ve BLOSUM62 gomulu tablolardan (dis bagimlilik YOK).
# Standart 20 AA disindaki degerler (*, indel kalintilari) ayri flag'lerle yakalanir.

STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")

# Grantham (1974) mesafe matrisi — yan zincir kompozisyon/polarite/hacim farki.
# Simetrik; sadece ust-ucgen tutulur, lookup iki yonlu yapilir.
_GRANTHAM = {
 ('S','R'):110,('S','L'):145,('S','P'):74,('S','T'):58,('S','A'):99,('S','V'):124,
 ('S','G'):56,('S','I'):142,('S','F'):155,('S','Y'):144,('S','C'):112,('S','H'):89,
 ('S','Q'):68,('S','N'):46,('S','K'):121,('S','D'):65,('S','E'):80,('S','M'):135,('S','W'):177,
 ('R','L'):102,('R','P'):103,('R','T'):71,('R','A'):112,('R','V'):96,('R','G'):125,('R','I'):97,
 ('R','F'):97,('R','Y'):77,('R','C'):180,('R','H'):29,('R','Q'):43,('R','N'):86,('R','K'):26,
 ('R','D'):96,('R','E'):54,('R','M'):91,('R','W'):101,
 ('L','P'):98,('L','T'):92,('L','A'):96,('L','V'):32,('L','G'):138,('L','I'):5,('L','F'):22,
 ('L','Y'):36,('L','C'):198,('L','H'):99,('L','Q'):113,('L','N'):153,('L','K'):107,('L','D'):172,
 ('L','E'):138,('L','M'):15,('L','W'):61,
 ('P','T'):38,('P','A'):27,('P','V'):68,('P','G'):42,('P','I'):95,('P','F'):114,('P','Y'):110,
 ('P','C'):169,('P','H'):77,('P','Q'):76,('P','N'):91,('P','K'):103,('P','D'):108,('P','E'):93,
 ('P','M'):87,('P','W'):147,
 ('T','A'):58,('T','V'):69,('T','G'):59,('T','I'):89,('T','F'):103,('T','Y'):92,('T','C'):149,
 ('T','H'):47,('T','Q'):42,('T','N'):65,('T','K'):78,('T','D'):85,('T','E'):65,('T','M'):81,('T','W'):128,
 ('A','V'):64,('A','G'):60,('A','I'):94,('A','F'):113,('A','Y'):112,('A','C'):195,('A','H'):86,
 ('A','Q'):91,('A','N'):111,('A','K'):106,('A','D'):126,('A','E'):107,('A','M'):84,('A','W'):148,
 ('V','G'):109,('V','I'):29,('V','F'):50,('V','Y'):55,('V','C'):192,('V','H'):84,('V','Q'):96,
 ('V','N'):133,('V','K'):97,('V','D'):152,('V','E'):121,('V','M'):21,('V','W'):88,
 ('G','I'):135,('G','F'):153,('G','Y'):147,('G','C'):159,('G','H'):98,('G','Q'):87,('G','N'):80,
 ('G','K'):127,('G','D'):94,('G','E'):98,('G','M'):127,('G','W'):184,
 ('I','F'):21,('I','Y'):33,('I','C'):198,('I','H'):94,('I','Q'):109,('I','N'):149,('I','K'):102,
 ('I','D'):168,('I','E'):134,('I','M'):10,('I','W'):61,
 ('F','Y'):22,('F','C'):205,('F','H'):100,('F','Q'):116,('F','N'):158,('F','K'):102,('F','D'):177,
 ('F','E'):140,('F','M'):28,('F','W'):40,
 ('Y','C'):194,('Y','H'):83,('Y','Q'):99,('Y','N'):143,('Y','K'):85,('Y','D'):160,('Y','E'):122,
 ('Y','M'):36,('Y','W'):37,
 ('C','H'):174,('C','Q'):154,('C','N'):139,('C','K'):202,('C','D'):154,('C','E'):170,('C','M'):196,('C','W'):215,
 ('H','Q'):24,('H','N'):68,('H','K'):32,('H','D'):81,('H','E'):40,('H','M'):87,('H','W'):115,
 ('Q','N'):46,('Q','K'):53,('Q','D'):61,('Q','E'):29,('Q','M'):101,('Q','W'):130,
 ('N','K'):94,('N','D'):23,('N','E'):42,('N','M'):142,('N','W'):174,
 ('K','D'):101,('K','E'):56,('K','M'):95,('K','W'):110,
 ('D','E'):45,('D','M'):160,('D','W'):181,
 ('E','M'):126,('E','W'):152,
 ('M','W'):67,
}
def grantham(a, b):
    if a == b: return 0
    return _GRANTHAM.get((a, b)) or _GRANTHAM.get((b, a))

# BLOSUM62 (sadece kullanilan ust-ucgen + kosegen). Eksikse 0 dondurulur.
from itertools import combinations_with_replacement as _cwr
_B62_RAW = '''A4 R-1 N-2 D-2 C0 Q-1 E-1 G0 H-2 I-1 L-1 K-1 M-1 F-2 P-1 S1 T0 W-3 Y-2 V0
R5 N0 D-2 C-3 Q1 E0 G-2 H0 I-3 L-2 K2 M-1 F-3 P-2 S-1 T-1 W-3 Y-2 V-3
N6 D1 C-3 Q0 E0 G0 H1 I-3 L-3 K0 M-2 F-3 P-2 S1 T0 W-4 Y-2 V-3
D6 C-3 Q0 E2 G-1 H-1 I-3 L-4 K-1 M-3 F-3 P-1 S0 T-1 W-4 Y-3 V-3
C9 Q-3 E-4 G-3 H-3 I-1 L-1 K-3 M-1 F-2 P-3 S-1 T-1 W-2 Y-2 V-1
Q5 E2 G-2 H0 I-3 L-2 K1 M0 F-3 P-1 S0 T-1 W-2 Y-1 V-2
E5 G-2 H0 I-3 L-3 K1 M-2 F-3 P-1 S0 T-1 W-3 Y-2 V-2
G6 H-2 I-4 L-4 K-2 M-3 F-3 P-2 S0 T-2 W-2 Y-3 V-3
H8 I-3 L-3 K-1 M-2 F-1 P-2 S-1 T-2 W-2 Y2 V-3
I4 L2 K-3 M1 F0 P-3 S-2 T-1 W-3 Y-1 V3
L4 K-2 M2 F0 P-3 S-2 T-1 W-2 Y-1 V1
K5 M-1 F-3 P-1 S0 T-1 W-3 Y-2 V-2
M5 F0 P-2 S-1 T-1 W-1 Y-1 V1
F6 P-4 S-2 T-2 W1 Y3 V-1
P7 S-1 T-1 W-4 Y-3 V-2
S4 T1 W-3 Y-2 V-2
T5 W-2 Y-2 V0
W11 Y2 V-3
Y7 V-1
V4'''
_ORDER = list("ARNDCQEGHILKMFPSTWYV")
_B62 = {}
for ri, line in enumerate(_B62_RAW.strip().split("\n")):
    toks = line.split()
    row_aa = toks[0][0]
    vals = [toks[0][1:]] + toks[1:]
    for ci, tok in enumerate(vals):
        col_aa = _ORDER[ri + ci]
        v = int(tok[1:] if tok[0].isalpha() else tok)  # ilk char harf olabilir
        _B62[(row_aa, col_aa)] = v; _B62[(col_aa, row_aa)] = v
def blosum62(a, b):
    return _B62.get((a, b), 0)

def add_fe(df):
    # Ham df'e FE sutunlari ekler (leakage-free: satir-bazli, fit gerektirmez).
    out = df.copy()
    a1 = out["AA_1"].astype("object"); a2 = out["AA_2"].astype("object")
    out["fe_aa_stopgain"] = (a2 == "*").astype(int)
    def _nonstd(v):
        return 0 if (isinstance(v, str) and v in STANDARD_AA) else 1
    out["fe_aa_nonstandard"] = (a1.map(_nonstd) | a2.map(_nonstd)).astype(int)
    def _gr(r):
        x, y = r["AA_1"], r["AA_2"]
        if isinstance(x, str) and isinstance(y, str) and x in STANDARD_AA and y in STANDARD_AA:
            return grantham(x, y)
        return -1
    def _bl(r):
        x, y = r["AA_1"], r["AA_2"]
        if isinstance(x, str) and isinstance(y, str) and x in STANDARD_AA and y in STANDARD_AA:
            return blosum62(x, y)
        return 0
    out["fe_grantham"] = out.apply(_gr, axis=1).astype(float)
    out["fe_blosum62"] = out.apply(_bl, axis=1).astype(float)
    return out

# Log-transform adaylari: genis pozitif aralikli AL frekans sutunlari (skew yuksek).
# Train uzerinde tespit edilir (leakage-free); EK skorlarina (negatif/0-1) DOKUNULMAZ.
def detect_log_cols(train_df):
    cand = []
    for c in NUM_COLS_BASE:
        s = pd.to_numeric(train_df[c], errors="coerce").dropna()
        if len(s) < 10: continue
        if s.min() >= 0 and s.max() > 1.0 and s.skew() > 2.0:
            cand.append(c)
    return cand

FE_NEW_COLS = ["fe_aa_stopgain", "fe_aa_nonstandard", "fe_grantham", "fe_blosum62"]
print(f"FE fonksiyonlari hazir. Yeni sutunlar: {FE_NEW_COLS} (+ al_freq_log dinamik)")
# Hizli dogrulama
_t = add_fe(master.head(50))
print("stopgain say:", int(_t['fe_aa_stopgain'].sum()),
      " nonstandard say:", int(_t['fe_aa_nonstandard'].sum()),
      " grantham ornek:", _t['fe_grantham'].head(3).tolist())


FE fonksiyonlari hazir. Yeni sutunlar: ['fe_aa_stopgain', 'fe_aa_nonstandard', 'fe_grantham', 'fe_blosum62'] (+ al_freq_log dinamik)
stopgain say: 0  nonstandard say: 9  grantham ornek: [43.0, 46.0, -1.0]


In [5]:
# Cell 5: M3 Preprocessing (fit-on-train) + FE entegrasyonu
AA_UNK = CR.AA_UNKNOWN_TOKEN

def fit_preprocessor(train_df_raw):
    # FE ekle, sonra M3 fit: log-cols, medyan, flag-source. Yalniz train.
    tr = add_fe(train_df_raw)
    log_cols = detect_log_cols(tr)
    num_cols = NUM_COLS_BASE + FE_NEW_COLS + [f"{c}__log" for c in log_cols]
    # log sutunlarini uret (orijinal de korunur -> M3 mantigi)
    for c in log_cols:
        tr[f"{c}__log"] = np.log1p(pd.to_numeric(tr[c], errors="coerce").clip(lower=0))
    miss = tr[base_feature_cols].isna().mean()
    flag_source = miss[miss > HIGH_MISSING_THRESHOLD].index.tolist()
    median = {c: pd.to_numeric(tr[c], errors="coerce").median() for c in num_cols}
    return {"log_cols": log_cols, "num_cols": num_cols, "cat_cols": CAT_LIKE,
            "flag_source": flag_source, "median": median}

def transform_X(df_raw, pp):
    df = add_fe(df_raw)
    for c in pp["log_cols"]:
        df[f"{c}__log"] = np.log1p(pd.to_numeric(df[c], errors="coerce").clip(lower=0))
    out = pd.DataFrame(index=df.index)
    for c in pp["num_cols"]:
        out[c] = pd.to_numeric(df[c], errors="coerce").fillna(pp["median"][c]).astype(float).values
    for c in pp["cat_cols"]:
        fill = AA_UNK if c in CR.AA_COLS else "MISSING"
        out[c] = df[c].astype("object").where(~df[c].isna(), fill).astype(str).values
    for c in pp["flag_source"]:
        out[CR.get_missing_mask_col_name(c)] = df[c].isna().astype(int).values
    return out, list(pp["cat_cols"])

_pp = fit_preprocessor(master)
_X, _cc = transform_X(master, _pp)
print(f"FE+M3 sonrasi: {_X.shape[1]} sutun  (log-cols: {len(_pp['log_cols'])}, "
      f"flag: {len(_pp['flag_source'])}, cat: {len(_cc)})  NaN kaldi mi: {_X.isna().any().any()}")


FE+M3 sonrasi: 432 sutun  (log-cols: 0, flag: 140, cat: 7)  NaN kaldi mi: False


In [6]:
# Cell 6: Split (NB15 ile ayni) + Degerlendirme Altyapisi (NB15 v2'den)
def panel_5050_split(df):
    pos = df[df[TARGET] == 1].sample(frac=1.0, random_state=SEED)
    neg = df[df[TARGET] == 0].sample(frac=1.0, random_state=SEED)
    npos, nneg = int(round(len(pos)*PANEL_SPLIT_FRAC)), int(round(len(neg)*PANEL_SPLIT_FRAC))
    tr = pd.concat([pos.iloc[:npos], neg.iloc[:nneg]]).sample(frac=1.0, random_state=SEED)
    te = pd.concat([pos.iloc[npos:], neg.iloc[nneg:]]).sample(frac=1.0, random_state=SEED)
    return tr.reset_index(drop=True), te.reset_index(drop=True)

def make_master_core(npos, nneg):
    pos = master[master[TARGET]==1].sample(n=min(npos,(master[TARGET]==1).sum()), random_state=SEED)
    neg = master[master[TARGET]==0].sample(n=min(nneg,(master[TARGET]==0).sum()), random_state=SEED)
    return pd.concat([pos, neg]).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

MASTER_CORE = make_master_core(MASTER_CORE_POS, MASTER_CORE_NEG)
PANEL_SPLITS = {p: panel_5050_split(panels[p]) for p in PANELS}
def build_s1(panel_train): return pd.concat([panel_train, MASTER_CORE], ignore_index=True)
for p in PANELS:
    tr, te = PANEL_SPLITS[p]
    print(f"{p}: train={len(tr)} (P={int((tr[TARGET]==1).sum())},B={int((tr[TARGET]==0).sum())}) "
          f"test={len(te)} (P={int((te[TARGET]==1).sum())},B={int((te[TARGET]==0).sum())})")

def _f1_pos(y, p): return f1_score(y, p, pos_label=1, zero_division=0)
def _metrics_at(y, prob, thr):
    yp = (prob >= thr).astype(int)
    return {"f1": _f1_pos(y, yp), "precision": precision_score(y, yp, pos_label=1, zero_division=0),
            "recall": recall_score(y, yp, pos_label=1, zero_division=0),
            "mcc": matthews_corrcoef(y, yp) if len(np.unique(y))>1 else 0.0}
def _resample_8020(y, prob, rng):
    y=np.asarray(y); prob=np.asarray(prob)
    neg=np.where(y==0)[0]; pos=np.where(y==1)[0]
    if len(neg)==0 or len(pos)==0: return y, prob
    npos=max(1, int(round(len(neg)*(1-FINAL_BENIGN_FRAC)/FINAL_BENIGN_FRAC)))
    keep=np.concatenate([neg, rng.choice(pos, size=npos, replace=True)])
    return y[keep], prob[keep]
def select_threshold(y_tr, p_tr, mode="f1_8020"):
    y_tr=np.asarray(y_tr); p_tr=np.asarray(p_tr)
    if mode=="f1_raw":
        thr,_=optimize_threshold(y_tr, p_tr); return float(thr)
    rng=np.random.RandomState(BOOT_SEED); yb,pb=_resample_8020(y_tr,p_tr,rng)
    best_thr,best=0.5,-2.0
    for thr in np.arange(0.05,0.95,0.01):
        yp=(pb>=thr).astype(int)
        s=_f1_pos(yb,yp) if mode=="f1_8020" else (matthews_corrcoef(yb,yp) if len(np.unique(yb))>1 else 0.0)
        if s>best: best,best_thr=s,thr
    return float(best_thr)
def bootstrap_8020(y_te, p_te, thr, n=N_BOOT):
    rng=np.random.RandomState(BOOT_SEED); f1s=[]
    for _ in range(n):
        yb,pb=_resample_8020(y_te,p_te,rng); f1s.append(_f1_pos(yb,(pb>=thr).astype(int)))
    f1s=np.array(f1s)
    return {"f1_8020_mean":float(f1s.mean()),"f1_8020_std":float(f1s.std()),
            "f1_8020_lo":float(np.percentile(f1s,2.5)),"f1_8020_hi":float(np.percentile(f1s,97.5))}
def evaluate(y_tr, p_tr, y_te, p_te, mode="f1_8020"):
    thr=select_threshold(y_tr,p_tr,mode)
    return {"thr":thr,"train":_metrics_at(y_tr,p_tr,thr),"test":_metrics_at(y_te,p_te,thr),
            "boot8020":bootstrap_8020(y_te,p_te,thr),
            "y_true":np.asarray(y_te),"y_pred":(np.asarray(p_te)>=thr).astype(int)}
print("Split + degerlendirme altyapisi hazir.")


CFTR: train=55 (P=45,B=10) test=56 (P=45,B=11)
KANSER: train=192 (P=132,B=60) test=193 (P=133,B=60)
PAH: train=185 (P=154,B=31) test=184 (P=153,B=31)
Split + degerlendirme altyapisi hazir.


In [7]:
from copy import deepcopy
# Cell 7: Base Model OOF Ureticileri (scratch: lgbm/catboost/nn/dnn + finetune: nn_ft/dnn_ft)
# Her base: train uzerinde N_OOF-fold OOF olasilik (leakage-free) + full-train -> test olasiligi.
# NN/DNN: SmallMLP + FocalLoss + early stopping (NB15 v2 ile ayni felsefe).
# Finetune base: MASTER cekirdek ile pretrain -> panel-train ile finetune. OOF'ta pretrain
#   BIR KEZ yapilir (fold-bagimsiz, MASTER'dan gelir, panel test'e degmez -> leakage minimal),
#   finetune fold'a ozgudur. Bu pragmatik sadelestirme rapora not edildi.

def _cv_n(y, want=N_OOF):
    y=np.asarray(y); mc=int(min((y==0).sum(),(y==1).sum()))
    return max(2, min(want, mc)) if mc>=2 else 2

class SmallMLP(torch.nn.Module):
    def __init__(self, d, hidden, n_layers, dropout):
        super().__init__(); layers=[]; inp=d
        for _ in range(n_layers):
            layers += [torch.nn.Linear(inp,hidden), torch.nn.ReLU(), torch.nn.Dropout(dropout)]; inp=hidden
        layers += [torch.nn.Linear(inp,1)]; self.net=torch.nn.Sequential(*layers)
    def forward(self,x): return self.net(x).squeeze(-1)

def _lgbm_fit_predict(Xtr, ytr, cat_cols, Xpred):
    m=lgb.LGBMClassifier(**{**LGBM_FIXED, "n_estimators":200, "num_leaves":31, "learning_rate":0.05})
    m.fit(Xtr, ytr, categorical_feature=cat_cols)
    return m.predict_proba(Xpred)[:,1], m
def _cb_fit_predict(Xtr, ytr, cat_idx, Xpred):
    m=CatBoostClassifier(**{**CB_FIXED, "iterations":200, "depth":4, "learning_rate":0.05})
    m.fit(Xtr, ytr, cat_features=cat_idx, silent=True)
    return m.predict_proba(Xpred)[:,1], m

def _prep_tree(train_df, test_df, pp):
    Xtr, cat_cols = transform_X(train_df, pp); Xte,_ = transform_X(test_df, pp)
    return Xtr.reset_index(drop=True), Xte.reset_index(drop=True), cat_cols

def _nn_encode(train_df, pp):
    Xtr, cat_cols = transform_X(train_df, pp)
    le_maps={}; Xn=Xtr.copy()
    for c in cat_cols:
        le=LabelEncoder(); Xn[c]=le.fit_transform(Xn[c].astype(str)); le_maps[c]={v:i for i,v in enumerate(le.classes_)}
    Xn=Xn.astype(np.float32); sc=StandardScaler().fit(Xn.values)
    return {"pp":pp,"cat_cols":cat_cols,"le_maps":le_maps,"scaler":sc,"columns":list(Xn.columns)}
def _nn_matrix(df, enc):
    X,_=transform_X(df, enc["pp"]); Xn=X.copy()
    for c in enc["cat_cols"]: Xn[c]=Xn[c].astype(str).map(enc["le_maps"][c]).fillna(-1)
    return torch.FloatTensor(enc["scaler"].transform(Xn[enc["columns"]].astype(np.float32).values))

def _make_mlp(kind, d):
    hidden, nl, dp = (128,2,0.4) if kind.startswith("nn") else (128,3,0.5)
    return SmallMLP(d, hidden, nl, dp)
def _train_es(model, X, y, lr=1e-3, max_epochs=60, patience=10):
    yv=np.asarray(y); strat=yv if min((yv==0).sum(),(yv==1).sum())>=2 else None
    tri,vai=train_test_split(np.arange(len(yv)),test_size=0.25,random_state=SEED,stratify=strat)
    pw=torch.FloatTensor([(yv[tri]==0).sum()/max((yv[tri]==1).sum(),1)])
    crit=FocalLoss(alpha=0.25, gamma=2.0, pos_weight=pw)
    opt=torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)
    Xt=X[tri]; yt=torch.FloatTensor(yv[tri].astype(np.float32)); Xvl=X[vai]; yvl=yv[vai]
    n=len(Xt); bs=min(64,max(2,n-1)); best,state,pat=-1,None,0
    for _ in range(max_epochs):
        model.train(); perm=torch.randperm(n)
        for i in range(0,n,bs):
            idx=perm[i:i+bs]
            if len(idx)<2: continue
            opt.zero_grad(); loss=crit(model(Xt[idx]),yt[idx]); loss.backward(); opt.step()
        model.eval()
        with torch.no_grad(): vp=torch.sigmoid(model(Xvl)).numpy()
        f=_f1_pos(yvl,(vp>=0.5).astype(int))
        if f>best: best,state,pat=f,deepcopy(model.state_dict()),0
        else:
            pat+=1
            if pat>=patience: break
    if state is not None: model.load_state_dict(state)
    return model
def _nn_proba(model, X):
    model.eval()
    with torch.no_grad(): return torch.sigmoid(model(X)).numpy()

def oof_and_test(train_df, test_df, base, pp, panel_train_df=None):
    # base: lightgbm/catboost/nn/dnn (scratch) | nn_ft/dnn_ft (finetune).
    # Scratch: train_df = build_s1(panel) (panel + MASTER cekirdek).
    # Finetune: pretrain=MASTER_CORE, finetune=panel_train_df. train_df burada
    #   meta hizalamasi icin build_s1 satir sirasiyla AYNI olmali -> oof yine build_s1 uzerinde.
    y=train_df[TARGET].values; n=len(train_df); oof=np.zeros(n); ns=_cv_n(y)
    skf=StratifiedKFold(n_splits=ns, shuffle=True, random_state=SEED)
    if base in ("lightgbm","catboost"):
        Xtr,Xte,cat_cols=_prep_tree(train_df, test_df, pp)
        if base=="lightgbm":
            for c in cat_cols:
                Xtr[c]=Xtr[c].astype("category"); Xte[c]=pd.Categorical(Xte[c],categories=Xtr[c].cat.categories)
            for tri,vai in skf.split(Xtr,y):
                pv,_=_lgbm_fit_predict(Xtr.iloc[tri], y[tri], cat_cols, Xtr.iloc[vai]); oof[vai]=pv
            ptest,_=_lgbm_fit_predict(Xtr, y, cat_cols, Xte)
        else:
            cat_idx=[list(Xtr.columns).index(c) for c in cat_cols]
            for tri,vai in skf.split(Xtr,y):
                pv,_=_cb_fit_predict(Xtr.iloc[tri], y[tri], cat_idx, Xtr.iloc[vai]); oof[vai]=pv
            ptest,_=_cb_fit_predict(Xtr, y, cat_idx, Xte)
        return oof, ptest
    elif base in ("nn","dnn"):
        kind=base
        enc=_nn_encode(train_df, pp); Xtr=_nn_matrix(train_df, enc); Xte=_nn_matrix(test_df, enc)
        for tri,vai in skf.split(Xtr.numpy(), y):
            m=_train_es(_make_mlp(kind, Xtr.shape[1]), Xtr[tri], y[tri]); oof[vai]=_nn_proba(m, Xtr[vai])
        mfull=_train_es(_make_mlp(kind, Xtr.shape[1]), Xtr, y); ptest=_nn_proba(mfull, Xte)
        return oof, ptest
    else:  # nn_ft / dnn_ft  (finetune)
        kind="nn" if base=="nn_ft" else "dnn"
        # encoder pretrain+panel_train birlesiminde fit (NB15 finetune ile ayni)
        fit_basis=pd.concat([MASTER_CORE, panel_train_df], ignore_index=True)
        enc=_nn_encode(fit_basis, pp)
        Xpre=_nn_matrix(MASTER_CORE, enc); ypre=MASTER_CORE[TARGET].values
        Xpan=_nn_matrix(panel_train_df, enc); ypan=panel_train_df[TARGET].values
        Xtr_full=_nn_matrix(train_df, enc); Xte=_nn_matrix(test_df, enc)
        # OOF: pretrain BIR KEZ (fold-bagimsiz), finetune fold'a ozgu. train_df = build_s1 sirasi.
        base_model=_train_es(_make_mlp(kind, Xpre.shape[1]), Xpre, ypre)
        # train_df icindeki panel-train satirlari ilk len(panel_train_df) satir (build_s1 sirasi)
        npan=len(panel_train_df)
        for tri,vai in skf.split(Xtr_full.numpy(), y):
            m=deepcopy(base_model)
            m=_train_es(m, Xtr_full[tri], y[tri], lr=1e-4, max_epochs=30, patience=8)
            oof[vai]=_nn_proba(m, Xtr_full[vai])
        # final: pretrain -> finetune (panel)
        mfull=deepcopy(base_model)
        mfull=_train_es(mfull, Xpan, ypan, lr=1e-4, max_epochs=30, patience=8)
        ptest=_nn_proba(mfull, Xte)
        return oof, ptest

print("Base OOF uretici hazir: oof_and_test (6 base: scratch x4 + finetune x2)")


Base OOF uretici hazir: oof_and_test (6 base: scratch x4 + finetune x2)


In [8]:
# Cell 8: Ana Dongu — FE ablasyonu x panel x (6 base + 2 meta), 3 threshold modu
# fe_mode in {'no_fe','with_fe'}: fit_preprocessor FE'yi her zaman ekliyor; izole olcum icin
# 'no_fe'da FE sutunlarini transform sonrasi dusururuz (en az kod degisikligi).
BASES = ["lightgbm", "catboost", "nn", "dnn", "nn_ft", "dnn_ft"]
SCRATCH_BASES = ["lightgbm", "catboost", "nn", "dnn"]
THR_MODES = ["f1_raw", "f1_8020", "mcc_8020"]
FE_MODES = ["no_fe", "with_fe"]
from sklearn.linear_model import LogisticRegression
import lightgbm as _lgb_meta

rows = []; DETAIL = {}

# 'no_fe' icin: transform_X ciktisindan FE sutunlarini dusur (M3-only ile ayni feature uzayi)
FE_DERIVED = list(FE_NEW_COLS)  # fe_* (log-cols MASTER'da 0 -> ek dusurme gerekmez ama dinamik ele alinir)
def _strip_fe(X):
    fe_cols = [c for c in X.columns if c.startswith("fe_") or c.endswith("__log")]
    return X.drop(columns=fe_cols), fe_cols

def build_meta(oof_dict):
    M = np.column_stack([oof_dict[b] for b in BASES])
    return np.hstack([M, np.column_stack([M.mean(1), M.std(1)])])

def _record(fe_mode, panel, model_name, ytr, ptr, yte, pte):
    for mode in THR_MODES:
        res = evaluate(ytr, ptr, yte, pte, mode)
        b8 = res["boot8020"]; tr = res["train"]
        rows.append({"fe_mode": fe_mode, "panel": panel, "model": model_name, "thr_mode": mode,
                     "threshold": round(res["thr"],3),
                     "train_f1": tr["f1"], "train_precision": tr["precision"],
                     "train_recall": tr["recall"], "train_mcc": tr["mcc"],
                     "test_f1_5050": res["test"]["f1"], "test_precision_5050": res["test"]["precision"],
                     "test_recall_5050": res["test"]["recall"], "test_mcc_5050": res["test"]["mcc"],
                     "test_f1_8020_mean": b8["f1_8020_mean"], "test_f1_8020_std": b8["f1_8020_std"],
                     "test_f1_8020_lo": b8["f1_8020_lo"], "test_f1_8020_hi": b8["f1_8020_hi"],
                     "n_test": len(yte)})
        if mode == "f1_8020":
            DETAIL[(fe_mode, panel, model_name)] = res

# transform_X'i fe_mode'a gore saran yardimci: no_fe icin FE sutunlarini dusur.
# Base ureticiler transform_X kullaniyor; fe_mode'u global bir bayrakla yonetiyoruz.
import builtins
_ORIG_TRANSFORM = transform_X
def _transform_fe_aware(df_raw, pp):
    X, cat = _ORIG_TRANSFORM(df_raw, pp)
    if _FE_MODE_FLAG[0] == "no_fe":
        X, _ = _strip_fe(X)
    return X, cat
_FE_MODE_FLAG = ["with_fe"]
transform_X = _transform_fe_aware   # base ureticiler artik bunu kullanir

for fe_mode in FE_MODES:
    _FE_MODE_FLAG[0] = fe_mode
    print(f"\n########## FE MODU: {fe_mode} ##########")
    for panel in PANELS:
        p_tr, p_te = PANEL_SPLITS[panel]
        train_df = build_s1(p_tr)
        y_tr = train_df[TARGET].values; y_te = p_te[TARGET].values
        pp = fit_preprocessor(train_df)
        print(f"--- {panel} (train={len(train_df)}, test={len(p_te)}) ---")
        oof_tr, proba_te = {}, {}
        for b in BASES:
            if b in SCRATCH_BASES:
                oof_b, te_b = oof_and_test(train_df, p_te, b, pp)
            else:
                oof_b, te_b = oof_and_test(train_df, p_te, b, pp, panel_train_df=p_tr)
            oof_tr[b], proba_te[b] = oof_b, te_b
            _record(fe_mode, panel, b, y_tr, oof_b, y_te, te_b)
        # Meta-learner'lar: LR + LightGBM
        Mtr, Mte = build_meta(oof_tr), build_meta(proba_te)
        lr = LogisticRegression(C=1.0, class_weight="balanced", max_iter=1000, random_state=SEED)
        lr.fit(Mtr, y_tr)
        _record(fe_mode, panel, "stack_lr", y_tr, lr.predict_proba(Mtr)[:,1], y_te, lr.predict_proba(Mte)[:,1])
        gbm = _lgb_meta.LGBMClassifier(n_estimators=100, max_depth=3, learning_rate=0.1,
                                       class_weight="balanced", verbosity=-1, random_state=SEED)
        gbm.fit(Mtr, y_tr)
        _record(fe_mode, panel, "stack_gbm", y_tr, gbm.predict_proba(Mtr)[:,1], y_te, gbm.predict_proba(Mte)[:,1])
        s = DETAIL[(fe_mode, panel, "stack_lr")]
        print(f"  stack_lr: F1(50/50)={s['test']['f1']:.3f} F1(80/20)={s['boot8020']['f1_8020_mean']:.3f}")

transform_X = _ORIG_TRANSFORM  # geri yukle
results_df = pd.DataFrame(rows)
results_df.to_csv(os.path.join(RESULTS_DIR, "fe_stacking_results.csv"), index=False)
print(f"\nToplam {len(results_df)} satir -> fe_stacking_results.csv")



########## FE MODU: no_fe ##########
--- CFTR (train=1305, test=56) ---
  stack_lr: F1(50/50)=0.722 F1(80/20)=0.563
--- KANSER (train=1442, test=193) ---
  stack_lr: F1(50/50)=0.888 F1(80/20)=0.665
--- PAH (train=1435, test=184) ---
  stack_lr: F1(50/50)=0.900 F1(80/20)=0.513

########## FE MODU: with_fe ##########
--- CFTR (train=1305, test=56) ---
  stack_lr: F1(50/50)=0.750 F1(80/20)=0.682
--- KANSER (train=1442, test=193) ---
  stack_lr: F1(50/50)=0.903 F1(80/20)=0.716
--- PAH (train=1435, test=184) ---
  stack_lr: F1(50/50)=0.872 F1(80/20)=0.515

Toplam 144 satir -> fe_stacking_results.csv


In [9]:
# Cell 9: Sonuc Derleme — FE katkisi + stacking vs base
viz = results_df[results_df["thr_mode"]=="f1_8020"]
print("=== %80/20 F1 (f1_8020) — fe_mode x panel x model ===")
print(viz.pivot_table(index=["fe_mode","model"], columns="panel", values="test_f1_8020_mean").round(4).to_string())

print("\n=== FE KATKISI (with_fe - no_fe, %80/20 F1, model-ortalama) ===")
for panel in PANELS:
    a=viz[(viz.panel==panel)&(viz.fe_mode=="with_fe")]["test_f1_8020_mean"].mean()
    b=viz[(viz.panel==panel)&(viz.fe_mode=="no_fe")]["test_f1_8020_mean"].mean()
    print(f"  {panel}: with_fe={a:.4f}  no_fe={b:.4f}  delta={a-b:+.4f}")
glob_fe=viz[viz.fe_mode=="with_fe"]["test_f1_8020_mean"].mean()-viz[viz.fe_mode=="no_fe"]["test_f1_8020_mean"].mean()
print(f"  GENEL FE delta: {glob_fe:+.4f}  ({'FE faydali' if glob_fe>0 else 'FE faydasiz/zararli'})")

print("\n=== Stacking vs en iyi base (with_fe, %80/20 F1) ===")
wf=viz[viz.fe_mode=="with_fe"]
for panel in PANELS:
    sub=wf[wf.panel==panel]
    stacks=sub[sub.model.str.startswith("stack")].sort_values("test_f1_8020_mean",ascending=False).iloc[0]
    base=sub[~sub.model.str.startswith("stack")].sort_values("test_f1_8020_mean",ascending=False).iloc[0]
    print(f"  {panel}: en iyi stack={stacks['model']}({stacks['test_f1_8020_mean']:.4f})  "
          f"en iyi base={base['model']}({base['test_f1_8020_mean']:.4f})  "
          f"-> {'STACK' if stacks['test_f1_8020_mean']>=base['test_f1_8020_mean'] else 'BASE'}")

print("\n=== Meta-learner karsilastirmasi (with_fe, panel-ort %80/20 F1) ===")
print(wf[wf.model.isin(["stack_lr","stack_gbm"])].groupby("model")["test_f1_8020_mean"].mean().round(4).to_string())
print("\n=== Ortalama train-test F1 farki (overfit, with_fe) ===")
wf2=wf.copy(); wf2["gap"]=wf2["train_f1"]-wf2["test_f1_5050"]
print(wf2.groupby("model")["gap"].mean().round(4).to_string())


=== %80/20 F1 (f1_8020) — fe_mode x panel x model ===
panel                CFTR  KANSER     PAH
fe_mode model                            
no_fe   catboost   0.7640  0.7203  0.4732
        dnn        0.6526  0.6504  0.3546
        dnn_ft     0.5025  0.6221  0.4849
        lightgbm   0.6697  0.7040  0.5140
        nn         0.5414  0.6876  0.4212
        nn_ft      0.4702  0.5991  0.4255
        stack_gbm  0.5238  0.6828  0.4837
        stack_lr   0.5634  0.6655  0.5126
with_fe catboost   0.8140  0.6894  0.4684
        dnn        0.8520  0.6465  0.4468
        dnn_ft     0.5450  0.7046  0.4490
        lightgbm   0.5802  0.6666  0.4686
        nn         0.5805  0.6378  0.3914
        nn_ft      0.6610  0.6835  0.4377
        stack_gbm  0.5227  0.7050  0.4703
        stack_lr   0.6820  0.7164  0.5151

=== FE KATKISI (with_fe - no_fe, %80/20 F1, model-ortalama) ===
  CFTR: with_fe=0.6547  no_fe=0.5860  delta=+0.0687
  KANSER: with_fe=0.6812  no_fe=0.6665  delta=+0.0148
  PAH: with_fe=0.45

In [10]:
# Cell 10: Gorseller
viz = results_df[results_df["thr_mode"]=="f1_8020"].copy()
MODEL_ORDER=["lightgbm","catboost","nn","dnn","nn_ft","dnn_ft","stack_lr","stack_gbm"]

# FIG1: with_fe panel x model
fig,ax=plt.subplots(figsize=(14,6))
wf=viz[viz.fe_mode=="with_fe"]
piv=wf.pivot_table(index="panel",columns="model",values="test_f1_8020_mean").reindex(PANELS)[MODEL_ORDER]
piv.plot(kind="bar",ax=ax,width=0.82); ax.set_ylim(0,1.05); ax.set_ylabel("%80/20 pathogenic-F1")
ax.set_title("NB16 — Base vs Stacking (with_fe, %80/20 F1)",fontweight="bold")
ax.legend(fontsize=7,ncol=8); ax.grid(axis="y",alpha=0.3); ax.tick_params(axis="x",rotation=0)
plt.tight_layout(); fig.savefig(os.path.join(RESULTS_DIR,"fig1_base_vs_stack.png"),dpi=110,bbox_inches="tight"); plt.show()

# FIG2: FE katkisi (with_fe vs no_fe, panel-ort)
fig,ax=plt.subplots(figsize=(9,5))
fe_piv=viz.pivot_table(index="panel",columns="fe_mode",values="test_f1_8020_mean").reindex(PANELS)
fe_piv[["no_fe","with_fe"]].plot(kind="bar",ax=ax,width=0.7,color=["#999999","#2ca02c"])
ax.set_ylim(0,1.0); ax.set_ylabel("%80/20 F1 (model-ort)"); ax.tick_params(axis="x",rotation=0)
ax.set_title("NB16 — FE Katkisi (no_fe vs with_fe)",fontweight="bold"); ax.grid(axis="y",alpha=0.3)
plt.tight_layout(); fig.savefig(os.path.join(RESULTS_DIR,"fig2_fe_ablation.png"),dpi=110,bbox_inches="tight"); plt.show()

# FIG3: stacking confusion matrix (with_fe, en iyi stack/panel)
from sklearn.metrics import ConfusionMatrixDisplay
fig,axes=plt.subplots(1,len(PANELS),figsize=(6*len(PANELS),5))
if len(PANELS)==1: axes=[axes]
for ax,panel in zip(axes,PANELS):
    sub=wf[(wf.panel==panel)&(wf.model.str.startswith("stack"))].sort_values("test_f1_8020_mean",ascending=False)
    mdl=sub.iloc[0]["model"]; res=DETAIL[("with_fe",panel,mdl)]
    cm=confusion_matrix(res["y_true"],res["y_pred"],labels=[0,1])
    ConfusionMatrixDisplay(cm,display_labels=["Benign","Pathogenic"]).plot(ax=ax,colorbar=False,cmap="viridis")
    ax.set_title(f"{panel} {mdl}\n(thr={res['thr']:.2f}, F1_5050={res['test']['f1']:.3f})")
plt.suptitle("NB16 — En Iyi Stacking Confusion Matrix (with_fe)",fontweight="bold")
plt.tight_layout(); fig.savefig(os.path.join(RESULTS_DIR,"fig3_confusion.png"),dpi=110,bbox_inches="tight"); plt.show()


In [12]:
# Cell 11: PDF Rapor (train sonuclari dahil)
from fpdf import FPDF
from PIL import Image
class NB16Report(FPDF):
    def header(self):
        self.set_font("Helvetica","",9); self.set_text_color(90,90,90)
        self.cell(0,8,"NB16 - FE + Stacking | TEKNOFEST Genetik Varyant",align="C",ln=True); self.set_text_color(0,0,0)
    def footer(self):
        self.set_y(-15); self.set_font("Helvetica","",8); self.set_text_color(120,120,120); self.cell(0,10,f"Sayfa {self.page_no()}",align="C")
    def section(self,t):
        self.ln(2); self.set_fill_color(31,119,180); self.set_text_color(255,255,255); self.set_font("Helvetica","B",12)
        self.cell(0,9,f"  {t}",ln=True,fill=True); self.set_text_color(0,0,0); self.ln(2)
    def body(self,t): self.set_font("Helvetica","",10); self.multi_cell(0,5.5,t); self.ln(1)
    def table(self,h,d,w):
        self.set_font("Helvetica","B",8); self.set_fill_color(70,130,180); self.set_text_color(255,255,255)
        for x,wi in zip(h,w): self.cell(wi,7,str(x),border=1,align="C",fill=True)
        self.ln(); self.set_text_color(0,0,0); self.set_font("Helvetica","",8); fill=False
        for row in d:
            self.set_fill_color(235,240,248)
            for v,wi in zip(row,w): self.cell(wi,6,str(v),border=1,align="C",fill=fill)
            self.ln(); fill=not fill
    def uw(self): return self.w-self.l_margin-self.r_margin
    def fit_image(self,p,max_h=None):
        iw,ih=Image.open(p).size; w=self.uw(); h=w*ih/iw
        if max_h and h>max_h: h=max_h; w=h*iw/ih
        self.image(p,w=w,h=h)

pdf=NB16Report(); pdf.set_auto_page_break(auto=True,margin=18)
viz=results_df[results_df["thr_mode"]=="f1_8020"]; wf=viz[viz.fe_mode=="with_fe"]
pdf.add_page(); pdf.set_font("Helvetica","B",18); pdf.ln(6)
pdf.cell(0,12,"NB16: Feature Engineering + Stacking (v2)",align="C",ln=True)
pdf.set_font("Helvetica","",12); pdf.cell(0,8,"TEKNOFEST Saglikta Yapay Zeka | Genetik Varyant",align="C",ln=True)
pdf.cell(0,8,f"Tarih: {datetime.now():%Y-%m-%d %H:%M}",align="C",ln=True); pdf.ln(4)
pdf.section("Yaklasim")
pdf.body("NB15 S1 kurgu (panel %50 + 625/625 MASTER cekirdek) + M3 missing uzerine: "
         "(1) FE ABLASYONU: no_fe (sadece M3) vs with_fe (Grantham/BLOSUM62/stopgain/log) "
         "-> FE'nin net katkisi izole olculur. (2) 6 base: scratch(lgbm,catboost,nn,dnn) + "
         "finetune(nn_ft,dnn_ft). (3) 2 meta: Logistic Regression + LightGBM. "
         "Threshold 3 mod, test %50/50 + bootstrap %80/20. Birincil: %80/20 F1.")
pdf.section("Not: Finetune base OOF sadelestirmesi")
pdf.body("nn_ft/dnn_ft OOF'ta MASTER-pretrain BIR KEZ yapilir (fold-bagimsiz, panel test'e "
         "degmez -> leakage minimal), finetune fold'a ozgudur. Tam-saf OOF degil; pragmatik.")

# 1. FE katkisi tablosu
pdf.section("1. FE Katkisi (panel-ort %80/20 F1)")
fr=[]
for panel in PANELS:
    a=viz[(viz.panel==panel)&(viz.fe_mode=="with_fe")]["test_f1_8020_mean"].mean()
    b=viz[(viz.panel==panel)&(viz.fe_mode=="no_fe")]["test_f1_8020_mean"].mean()
    fr.append([panel,f"{b:.4f}",f"{a:.4f}",f"{a-b:+.4f}"])
pdf.table(["Panel","no_fe","with_fe","delta"],fr,[40,45,45,45])

# 2. with_fe %80/20 tablo (tum modeller)
pdf.section("2. with_fe - %80/20 F1 (panel x model)")
piv=wf.pivot_table(index="model",columns="panel",values="test_f1_8020_mean")
pdf.table(["Model"]+PANELS,[[m]+[f"{piv.loc[m,p]:.4f}" for p in PANELS] for m in piv.index],
          [44]+[44]*len(PANELS))

# 3. TRAIN sonuclari (overfit) — with_fe stacking + base ozet
pdf.add_page()
pdf.section("3. Train Sonuclari (with_fe, f1_8020) + Train-Test farki")
th=["Panel","Model","Tr-F1","Tr-P","Tr-R","Te-F1(50/50)","Tr-Te farki"]
tw=[24,28,20,18,18,28,26]
td=[]
for r in wf.sort_values(["panel","model"]).itertuples():
    td.append([r.panel, r.model, f"{r.train_f1:.3f}", f"{r.train_precision:.3f}",
               f"{r.train_recall:.3f}", f"{r.test_f1_5050:.3f}", f"{r.train_f1-r.test_f1_5050:+.3f}"])
pdf.table(th,td,tw)

# 4. Stacking vs base + meta karsilastirmasi
pdf.add_page()
pdf.section("4. Stacking vs En Iyi Base (with_fe, %80/20 F1)")
rs=[]
for panel in PANELS:
    sub=wf[wf.panel==panel]
    st=sub[sub.model.str.startswith("stack")].sort_values("test_f1_8020_mean",ascending=False).iloc[0]
    ba=sub[~sub.model.str.startswith("stack")].sort_values("test_f1_8020_mean",ascending=False).iloc[0]
    rs.append([panel,f"{st['model']} ({st['test_f1_8020_mean']:.4f})",
               f"{ba['model']} ({ba['test_f1_8020_mean']:.4f})",
               "stack" if st['test_f1_8020_mean']>=ba['test_f1_8020_mean'] else "base"])
pdf.table(["Panel","En iyi stack","En iyi base","Kazanan"],rs,[28,55,55,30])
pdf.ln(2); pdf.section("4b. Meta-learner (with_fe, panel-ort %80/20 F1)")
mm=wf[wf.model.isin(["stack_lr","stack_gbm"])].groupby("model")["test_f1_8020_mean"].mean()
pdf.table(["Meta","Ort. %80/20 F1"],[[m,f"{mm[m]:.4f}"] for m in mm.index],[60,60])

# grafikler
WIDE={"fig1_base_vs_stack.png","fig3_confusion.png"}
for fn,t in [("fig2_fe_ablation.png","5. FE Ablasyonu (no_fe vs with_fe)"),
             ("fig1_base_vs_stack.png","6. Base vs Stacking (with_fe)"),
             ("fig3_confusion.png","7. En Iyi Stacking Confusion Matrix")]:
    p=os.path.join(RESULTS_DIR,fn)
    if os.path.exists(p):
        pdf.add_page(orientation="L" if fn in WIDE else "P"); pdf.section(t)
        pdf.fit_image(p,max_h=pdf.h-pdf.get_y()-20)
out=os.path.join(REPORTS_DIR,"NB16_fe_stacking_report.pdf"); pdf.output(out)
print("PDF:",out); print("CSV:",os.path.join(RESULTS_DIR,"fe_stacking_results.csv"))


PDF: c:\Users\ahmet.ceyhan23\Desktop\teknofest_model\reports\NB16_fe_stacking_report.pdf
CSV: c:\Users\ahmet.ceyhan23\Desktop\teknofest_model\results\v6_fe_stacking\fe_stacking_results.csv
